In [1]:
import pandas as pd
import itertools
import networkx as nx
import collections
import os
import pyarrow.parquet as pq
import numpy as np
#import folium
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
countryDF = pd.read_csv('../data/Floriana_country_info.csv')
id2name=dict(zip(countryDF['alpha-2'],countryDF['name']))
id2region=dict(zip(countryDF['alpha-2'],countryDF['region']))
id2subregion=dict(zip(countryDF['alpha-2'],countryDF['sub-region']))

id2name['XK']='Kosovo'
id2region['XK']='Europe'
id2subregion['XK']='Southern Europe'

In [5]:
dfTopics=pd.read_csv('../data/Floriana_topic_mapping.csv')
dfTopics['topic_id']=dfTopics['topic_id'].apply(lambda x: 'T'+str(x))
id2name=dict(zip(dfTopics['topic_id'],dfTopics['topic_name']))
id2field=dict(zip(dfTopics['topic_id'],dfTopics['field_name']))

In [6]:
dfTopics

,topic_id,topic_name,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,T10001,Tectonic and Geochronological Evolution of Oro...,1908,Geophysics,19,Earth and Planetary Sciences,3,Physical Sciences,Zircon; Geochronology; Tectonics; Granitic Roc...,This cluster of papers focuses on the tectonic...,https://en.wikipedia.org/wiki/Geochronology
1,T10002,Advancements in Density Functional Theory,3107,"Atomic and Molecular Physics, and Optics",31,Physics and Astronomy,3,Physical Sciences,Density Functional Theory; Dispersion Correcti...,This cluster of papers represents advancements...,https://en.wikipedia.org/wiki/Density_function...
2,T10003,Knowledge Management and Organizational Innova...,1408,Strategy and Management,14,"Business, Management and Accounting",2,Social Sciences,Dynamic Capabilities; Knowledge Transfer; Busi...,This cluster of papers revolves around the top...,https://en.wikipedia.org/wiki/Knowledge_manage...
3,T10004,Soil Carbon Dynamics and Nutrient Cycling in E...,1111,Soil Science,11,Agricultural and Biological Sciences,1,Life Sciences,Soil Carbon Sequestration; Nitrogen Cycle; Mic...,This cluster of papers explores the dynamics o...,https://en.wikipedia.org/wiki/Soil_carbon_dyna...
4,T10005,Biodiversity Conservation and Ecosystem Manage...,2309,Nature and Landscape Conservation,23,Environmental Science,3,Physical Sciences,Biodiversity; Conservation; Ecosystem; Invasiv...,This cluster of papers focuses on the conserva...,https://en.wikipedia.org/wiki/Biodiversity_con...
...,...,...,...,...,...,...,...,...,...,...,...
4511,T14517,History of Science and Knowledge Production,1207,History and Philosophy of Science,12,Arts and Humanities,2,Social Sciences,History; Science; Knowledge Production; Medici...,This cluster of papers covers a wide range of ...,https://en.wikipedia.org/wiki/History_of_science
4512,T14518,Baseball's Influence on American Culture and S...,1202,History,12,Arts and Humanities,2,Social Sciences,Baseball; American Culture; Societal Influence...,This cluster of papers explores the profound i...,https://en.wikipedia.org/wiki/Baseball_in_the_...
4513,T14519,Digital Education and Knowledge Economy,1710,Information Systems,17,Computer Science,3,Physical Sciences,Digital Education; E-Learning; Knowledge Econo...,This cluster of papers explores the intersecti...,https://en.wikipedia.org/wiki/Digital_education
4514,T14520,Slow Cities Movement and Sustainable Urban Dev...,3322,Urban Studies,33,Social Sciences,2,Social Sciences,Slow Cities; Sustainable Development; Urban Su...,This cluster of papers explores the Slow Citie...,https://en.wikipedia.org/wiki/Citt%C3%A0slow


In [7]:
parqToRead='authorsPapersTopicsYearInstitutions.parquet'
# Read the parquet file filtering the lines where year=2005
table = pq.read_table(parqToRead, filters=[('year','=','2010')])
#table = pq.read_table(parqToRead)
# Convert to pandas DataFrame
df = table.to_pandas()


In [7]:
df

,authors,id,year,institution,country,coord,inst_name,topic
83,A5104829885,W4400863937,2010,I189238007,UA,"[50.45466, 30.5238]",Taras Shevchenko National University of Kyiv,T12618
84,A5010153016,W4400863937,2010,I189238007,UA,"[50.45466, 30.5238]",Taras Shevchenko National University of Kyiv,T12618
85,A5061654139,W4400863937,2010,I189238007,UA,"[50.45466, 30.5238]",Taras Shevchenko National University of Kyiv,T12618
820,A5052243613,W2112665120,2010,I4210134612,GB,"[51.1891, -0.890375]",Forest Research,T12618
821,A5072218476,W2112665120,2010,I4210134612,GB,"[51.1891, -0.890375]",Forest Research,T12618
...,...,...,...,...,...,...,...,...
616473764,A5004212789,W4253298579,2010,I88491126,RO,"[44.447567, 26.09665]",Bucharest University of Economic Studies,T13598
616473769,A5090501388,W4245646241,2010,I172574986,LT,"[54.90272, 23.90961]",Kaunas University of Technology,T13598
616473820,A5088839860,W4229960273,2010,I153976015,SI,"[46.05108, 14.50513]",University of Ljubljana,T13598
616473835,A5058403140,W4239836736,2010,I184889055,US,"[37.78402, -79.44282]",Washington and Lee University,T13598
